# Week 3: Better Prompts & Structured Output

**Lab companion to [week_03.md](../agenda/week_03.md)**

In this lab, you will:
1. Master prompt engineering techniques
2. Get reliable JSON output
3. Validate AI responses with Pydantic
4. Build extraction and classification systems

In [ ]:
# Setup
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv()
client = OpenAI()

print("✓ Ready!")

## 1. Prompt Engineering Basics

### Zero-Shot: Just ask directly

In [ ]:
# Zero-shot classification
text = "This product is amazing! Best purchase ever!"

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": f"Classify this text as positive, negative, or neutral: '{text}'"
    }]
)

print(response.choices[0].message.content)

### Few-Shot: Provide examples

In [ ]:
# Few-shot classification with examples
prompt = """Classify sentiment as positive, negative, or neutral.

Examples:
"I love this!" -> positive
"Terrible, don't buy" -> negative  
"It's okay, nothing special" -> neutral

Now classify: "This was a waste of money"
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

### Chain-of-Thought: Make AI explain reasoning

In [ ]:
# Chain-of-thought for better reasoning
prompt = """Solve this step by step:

A store has 50 apples. They sell 23 in the morning and receive a delivery of 35 more.
In the afternoon, they sell 18 and throw away 5 that went bad.
How many apples do they have at the end of the day?

Think through each step before giving the final answer.
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

## 2. Getting JSON Output

### Method 1: Ask for JSON (unreliable)

In [ ]:
# Just asking for JSON (sometimes fails)
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": "Extract the person's name and age from: 'John is 25 years old'. Return JSON."
    }]
)

print("Raw response:")
print(response.choices[0].message.content)

# Try to parse - might fail!
try:
    data = json.loads(response.choices[0].message.content)
    print("\nParsed:", data)
except json.JSONDecodeError as e:
    print(f"\nFailed to parse JSON: {e}")

### Method 2: JSON Mode (reliable)

In [ ]:
# Using response_format for guaranteed JSON
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": "Extract the person's name and age from: 'John is 25 years old'. Return as JSON with 'name' and 'age' fields."
    }],
    response_format={"type": "json_object"}
)

# Now it's always valid JSON
data = json.loads(response.choices[0].message.content)
print("Parsed:", data)
print(f"Name: {data['name']}, Age: {data['age']}")

## 3. Pydantic Validation

Make AI output match a specific schema:

In [ ]:
!pip install pydantic -q

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

# Define the schema
class Person(BaseModel):
    name: str = Field(description="The person's full name")
    age: int = Field(description="Age in years")
    occupation: Optional[str] = Field(default=None, description="Job title")

# Generate the schema as a string for the prompt
schema_str = Person.model_json_schema()
print("Schema:")
print(json.dumps(schema_str, indent=2))

In [ ]:
def extract_person(text: str) -> Person:
    """Extract person info from text using Pydantic validation."""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Extract person information from this text:
{text}

Return JSON matching this schema:
- name: string (full name)
- age: integer
- occupation: string or null"""
        }],
        response_format={"type": "json_object"}
    )
    
    data = json.loads(response.choices[0].message.content)
    return Person(**data)  # Validates and converts

# Test
person = extract_person("Dr. Sarah Johnson, a 42-year-old surgeon, won the award.")
print(f"Name: {person.name}")
print(f"Age: {person.age}")
print(f"Occupation: {person.occupation}")

## 4. Complex Extraction

Extract multiple entities with nested structures:

In [ ]:
from typing import List
from enum import Enum

class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

class Task(BaseModel):
    title: str
    priority: Priority
    assignee: Optional[str] = None

class MeetingNotes(BaseModel):
    title: str
    date: str
    attendees: List[str]
    tasks: List[Task]
    summary: str

def extract_meeting_notes(transcript: str) -> MeetingNotes:
    """Extract structured meeting notes from a transcript."""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Extract meeting notes from this transcript:

{transcript}

Return JSON with:
- title: meeting title
- date: date mentioned (or "unknown")
- attendees: list of participant names
- tasks: list of {{title, priority (low/medium/high), assignee}}
- summary: brief 1-2 sentence summary"""
        }],
        response_format={"type": "json_object"}
    )
    
    data = json.loads(response.choices[0].message.content)
    return MeetingNotes(**data)

# Test
transcript = """
Project sync meeting, March 15th.
Attendees: Alice, Bob, and Carol.

Alice mentioned the frontend is almost done, just needs testing.
Bob said the API has a critical bug that needs fixing ASAP - he'll handle it.
Carol will review the documentation by end of week, low priority.

Overall, we're on track for the launch.
"""

notes = extract_meeting_notes(transcript)
print(f"Meeting: {notes.title}")
print(f"Date: {notes.date}")
print(f"Attendees: {', '.join(notes.attendees)}")
print(f"\nTasks:")
for task in notes.tasks:
    print(f"  - [{task.priority.value}] {task.title} (assigned to: {task.assignee or 'unassigned'})")
print(f"\nSummary: {notes.summary}")

## 5. Classification with Confidence

Get AI to rate its confidence:

In [ ]:
class Classification(BaseModel):
    category: str
    confidence: float = Field(ge=0, le=1, description="Confidence score 0-1")
    reasoning: str

def classify_with_confidence(text: str, categories: List[str]) -> Classification:
    """Classify text into one of the categories with confidence."""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{
            "role": "user",
            "content": f"""Classify this text into one of these categories: {categories}

Text: {text}

Return JSON with:
- category: one of the categories above
- confidence: 0.0 to 1.0
- reasoning: brief explanation"""
        }],
        response_format={"type": "json_object"}
    )
    
    data = json.loads(response.choices[0].message.content)
    return Classification(**data)

# Test
texts = [
    "My order hasn't arrived yet, it's been 2 weeks!",
    "How do I reset my password?",
    "Your product is terrible and I want a refund",
    "Thanks for the quick delivery!"
]

categories = ["complaint", "question", "feedback", "other"]

for text in texts:
    result = classify_with_confidence(text, categories)
    print(f"'{text[:40]}...'")
    print(f"  → {result.category} ({result.confidence:.0%})")
    print(f"  Reason: {result.reasoning}")
    print()

## 6. Building a Data Extraction Pipeline

In [ ]:
class ContactInfo(BaseModel):
    name: Optional[str] = None
    email: Optional[str] = None
    phone: Optional[str] = None
    company: Optional[str] = None

class DataExtractor:
    """Extract structured data from unstructured text."""
    
    def __init__(self):
        self.client = OpenAI()
    
    def extract(self, text: str, model_class: type) -> BaseModel:
        """Generic extraction for any Pydantic model."""
        
        # Get field descriptions
        fields = []
        for name, field in model_class.model_fields.items():
            fields.append(f"- {name}: {field.annotation.__name__}")
        
        response = self.client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{
                "role": "user",
                "content": f"""Extract information from this text:

{text}

Return JSON with these fields (use null if not found):
{chr(10).join(fields)}"""
            }],
            response_format={"type": "json_object"}
        )
        
        data = json.loads(response.choices[0].message.content)
        return model_class(**data)

# Test
extractor = DataExtractor()

text = """Hi, this is Mark from TechCorp. 
You can reach me at mark.wilson@techcorp.com or call 555-123-4567."""

contact = extractor.extract(text, ContactInfo)
print(f"Name: {contact.name}")
print(f"Email: {contact.email}")
print(f"Phone: {contact.phone}")
print(f"Company: {contact.company}")

## 7. Exercises

In [ ]:
# Exercise 1: Create a recipe extractor
# Extract ingredients and steps from recipe text

class Recipe(BaseModel):
    name: str
    servings: int
    ingredients: List[str]
    steps: List[str]

recipe_text = """
Simple Pasta (serves 4)

You'll need: 400g spaghetti, 2 cloves garlic, olive oil, salt, pepper, parmesan.

First, boil water and cook pasta until al dente. Meanwhile, sauté minced garlic
in olive oil. Drain pasta, toss with garlic oil, season with salt and pepper.
Serve with grated parmesan.
"""

# Your code here - use the DataExtractor pattern
# recipe = ...

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Define a Pydantic model for recipes:
- name: str
- ingredients: List[str]
- steps: List[str]
- prep_time_minutes: int
- difficulty: Literal["easy", "medium", "hard"]

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
from pydantic import BaseModel
from typing import List, Literal

class Recipe(BaseModel):
    name: str
    ingredients: List[str]
    steps: List[str]
    prep_time_minutes: int
    difficulty: Literal["easy", "medium", "hard"]

def extract_recipe(text: str) -> Recipe:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract recipe information."},
            {"role": "user", "content": text}
        ],
        response_format=Recipe
    )
    return response.choices[0].message.parsed

recipe = extract_recipe("To make pasta: boil water, add pasta, cook 8 mins...")
print(recipe.model_dump_json(indent=2))
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

Define a Pydantic model for recipes:
- name: str
- ingredients: List[str]
- steps: List[str]
- prep_time_minutes: int
- difficulty: Literal["easy", "medium", "hard"]

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
from pydantic import BaseModel
from typing import List, Literal

class Recipe(BaseModel):
    name: str
    ingredients: List[str]
    steps: List[str]
    prep_time_minutes: int
    difficulty: Literal["easy", "medium", "hard"]

def extract_recipe(text: str) -> Recipe:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Extract recipe information."},
            {"role": "user", "content": text}
        ],
        response_format=Recipe
    )
    return response.choices[0].message.parsed

recipe = extract_recipe("To make pasta: boil water, add pasta, cook 8 mins...")
print(recipe.model_dump_json(indent=2))
```

</details>

In [ ]:
# Exercise 2: Multi-label classification
# Classify text into MULTIPLE categories

class MultiLabel(BaseModel):
    categories: List[str]
    confidence: float

def multi_classify(text: str):
    """Classify into multiple categories."""
    categories = ["urgent", "technical", "billing", "feedback", "general"]
    
    # Your implementation here
    pass

# Test with: "The app is crashing and I can't access my billing info! Please help ASAP!"

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

For multi-label, return a list of categories:
- Use List[str] in your model
- Or use a dict with boolean flags for each category

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ContentLabels(BaseModel):
    categories: List[str]
    is_urgent: bool
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float

def classify_content(text: str) -> ContentLabels:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Classify the content."},
            {"role": "user", "content": text}
        ],
        response_format=ContentLabels
    )
    return response.choices[0].message.parsed

result = classify_content("URGENT: Server down! Need help immediately!")
print(result)  # categories=['technical', 'support'], is_urgent=True, ...
```

</details>

<details>
<summary>💡 <b>Hint</b> (click to expand)</summary>

For multi-label, return a list of categories:
- Use List[str] in your model
- Or use a dict with boolean flags for each category

</details>

<details>
<summary>✅ <b>Solution</b> (click to expand)</summary>

```python
class ContentLabels(BaseModel):
    categories: List[str]
    is_urgent: bool
    sentiment: Literal["positive", "negative", "neutral"]
    confidence: float

def classify_content(text: str) -> ContentLabels:
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Classify the content."},
            {"role": "user", "content": text}
        ],
        response_format=ContentLabels
    )
    return response.choices[0].message.parsed

result = classify_content("URGENT: Server down! Need help immediately!")
print(result)  # categories=['technical', 'support'], is_urgent=True, ...
```

</details>

## Summary

You learned:
- ✅ Zero-shot, few-shot, and chain-of-thought prompting
- ✅ Getting reliable JSON with response_format
- ✅ Validating with Pydantic
- ✅ Complex extraction and classification

**Next:** [Week 4 - Connect to Your Data (RAG)](week_04_rag_intro.ipynb)